# Optimizing our Models

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

## Load and Process the Data
This is using the same procedure as in housing_prices.ipynb. It would make sense to extract this into its own distinct process so that we didn't have to duplicate the code.

In [2]:
df = pd.read_csv('./data/housing.csv')
df.dropna(inplace=True)

In [3]:
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42)

In [5]:
# Define the categories in the desired order for encoding
categories = [['INLAND', '<1H OCEAN', 'NEAR OCEAN', 'NEAR BAY', 'ISLAND']]
encoder = OrdinalEncoder(categories=categories)
X_train['ocean_proximity'] = encoder.fit_transform(
    X_train[['ocean_proximity']])

In [6]:
# Encode the ocean_proximity column as well
X_test['ocean_proximity'] = encoder.fit_transform(X_test[['ocean_proximity']])

## Optimize our Models with Grid Search
https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

In [7]:
# Define the models and parameters for grid search
models_params = {
    'DecisionTree': {
        'model': DecisionTreeRegressor(random_state=42),
        'params': {
            'max_depth': [5, 10, 15, 20],
            'min_samples_split': [2, 5, 10]
        }
    },
    'RandomForest': {
        'model': RandomForestRegressor(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_features': ['sqrt', 'log2']
        }
    },
    'HistGradientBoosting': {
        'model': HistGradientBoostingRegressor(random_state=42),
        'params': {
            'max_iter': [50, 100, 150],
            'learning_rate': [0.01, 0.1, 0.2]
        }
    },
    'XGBoost': {
        'model': xgb.XGBRegressor(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.1, 0.2],
            'max_depth': [3, 6, 9],
            'subsample': [0.5, 0.7, 1.0],
            'colsample_bytree': [0.5, 0.7, 1.0]
        }
    }
}

# Setup grid search and evaluate each model
results = {}
for name, setup in models_params.items():
    grid_search = GridSearchCV(
        setup['model'], setup['params'], cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    results[name] = {
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_
    }

results

{'DecisionTree': {'best_score': -4135742706.403236,
  'best_params': {'max_depth': 10, 'min_samples_split': 10}},
 'RandomForest': {'best_score': -2576306405.113396,
  'best_params': {'max_features': 'sqrt', 'n_estimators': 300}},
 'HistGradientBoosting': {'best_score': -2300070904.601216,
  'best_params': {'learning_rate': 0.2, 'max_iter': 150}},
 'XGBoost': {'best_score': -2258141391.9542584,
  'best_params': {'colsample_bytree': 1.0,
   'learning_rate': 0.1,
   'max_depth': 6,
   'n_estimators': 300,
   'subsample': 0.7}}}

## Activity: Perform a Grid Search
Compare the results of using this grid search approach to tune hyperparameters with the results of the corresponding models in the housing_prices.ipynb notebook. Did we improve on the results for all classifiers?

Actually, RandomForest and HistGradientBoosting are worse.

<table>
    <tr>
        <td></td>             <td>DecisionTree</td> <td>RandomForest</td> <td>HistGradientBoosting</td> <td>XGBoost</td>
    </tr>
    <tr>
        <td>Previous MSE</td> <td>4517192285</td>   <td>2370980682</td>   <td>2276617427</td>           <td>2502898859</td>
    </tr>
    <tr>
        <td>Grid search</td>  <td>4135742706</td>   <td>2576306405</td>   <td>2300070904</td>           <td>2258141391</td>
    </tr>
</table>

This suggests that we didn't configure our search to explore thoroughly enough. Examine the hyperperameters for each model type and set up the grid search to explore an appropriate set of hyperparameters and value ranges.
 - https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html
 - https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html
 - https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html
 - https://xgboost.readthedocs.io/en/stable/python/python_api.html

Try to ensure that the grid search improves on, or at least equals, the previous results for RandomForest and HistGradientBoosting.

The MSE values returned by GridSearchCV refer to the training data. Using the optimized hyperparameters, create instances of these regressors and evaluate them in terms of MSE on the test data.

__CAUTION__: The goal is to get familiar with hyperparameter tuning, not to use electricity for hours and hours training models. It is easy to get carried away!

Optional: Scikit-learn has other hyperparameter optimizers and there are additional approaches provided by other modules. You may find that you get better results more quickly with those, or see other notebooks where the alternative approaches are used. 

## Activity: Examine Other Solutions
Examine the solutions that other people have developed for the California Housing Prices dataset with a view to understanding how they have tried to improve their models. For example, user OMARAYMANATIA achieves an MSE of 48911 at https://www.kaggle.com/code/omaraymanatia/california-housing-prices-prediction.

Kaggle conveniently shows you the rating of users (OMARAYMANATIA is a rated Kaggle Expert because of his notebooks) so we can focus on the work of the more experienced Kaggle users.

From your explorations, compile a list of questions and ideas to try. Sort these by priority, taking into account how much work is involved (effort) and estimated likely payoff (value). Low-effort high-value actions are a great place to continue building your knowledge and skills. What list did you come up with?

